## Cell 1 — Imports and Label Loading
Load all required libraries and read the labels CSV files generated
by the topomap pipeline. These contain metadata (participant ID, 
treatment ID, loop number) and targets (arousal, heart rate, GSR)
for every row in the dataset.

In [1]:
import os, json, time

# ---- Change this ONE line for each new experiment ----
EXPERIMENT = "step3_optuna_tuning"
DESCRIPTION = "step1 features (loop onehot + videoID emb) + MSE + 1.0*firstdiff loss, eval=plain masked RMSE, 128x128 +optuna tuning"

os.makedirs("models", exist_ok=True)
BEST_MODEL_PATH = f"models/{EXPERIMENT}_best.pt"
RESULTS_LOG = "models/results_log.jsonl"

In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import joblib
import os

# ---- Paths ----
TOPOMAP_DIR = Path('topomap_data')
TRAIN_LABELS = TOPOMAP_DIR / 'train' / 'labels.csv'
VAL_LABELS = TOPOMAP_DIR / 'val' / 'labels.csv'

# ---- Load labels ----
train_labels = pd.read_csv(TRAIN_LABELS)
val_labels = pd.read_csv(VAL_LABELS)

print("Train samples:", len(train_labels))
print("Val samples:", len(val_labels))
print("\nColumns:", train_labels.columns.tolist())
print("\nSample:")
print(train_labels.head())

Train samples: 244620
Val samples: 83400

Columns: ['filename', 'row_idx', 'participant_id', 'treatment_id', 'loop_num', 'bin_within_loop', 'arousal', 'heartrate', 'gsr']

Sample:
   filename  row_idx participant_id treatment_id  loop_num  bin_within_loop  \
0  P001_T01        0           P001          T01         1                1   
1  P001_T01        1           P001          T01         1                2   
2  P001_T01        2           P001          T01         1                3   
3  P001_T01        3           P001          T01         1                4   
4  P001_T01        4           P001          T01         1                5   

   arousal   heartrate       gsr  
0   0.0000  112.496268  0.000010  
1   0.0000  114.324143  0.000010  
2   0.0000  115.561028  0.000011  
3   0.0008  115.587845  0.000011  
4   0.0020  115.910654  0.000011  


In [3]:
# Map treatment IDs to integer indices (T01 -> 0, T02 -> 1, ...)
all_treatments = sorted(train_labels['treatment_id'].unique())
treatment_to_idx = {t: i for i, t in enumerate(all_treatments)}
NUM_TREATMENTS = len(treatment_to_idx)
print(f"{NUM_TREATMENTS} treatments mapped")

24 treatments mapped


## Cell 2 — Fit StandardScaler on Train Data
Fit StandardScaler on training set physiological features and labels.
Scaler is fit on train only then applied to both train and val to prevent data leakage.
Fitted scalers are saved to disk for later use during inference.

In [4]:
os.makedirs('scalers', exist_ok=True)

# ---- Physiological features to scale ----
phys_cols = ['heartrate', 'gsr']

# ---- Target columns to scale ----
target_cols = ['arousal', 'heartrate', 'gsr']

# ---- Fit scalers on train only ----
scaler_phys = StandardScaler()
scaler_arousal = StandardScaler()
scaler_hr = StandardScaler()
scaler_gsr = StandardScaler()

# Fit on train
scaler_phys.fit(train_labels[phys_cols])
scaler_arousal.fit(train_labels[['arousal']])
scaler_hr.fit(train_labels[['heartrate']])
scaler_gsr.fit(train_labels[['gsr']])

# Save scalers
joblib.dump(scaler_phys, 'scalers/scaler_phys.pkl')
joblib.dump(scaler_arousal, 'scalers/scaler_arousal.pkl')
joblib.dump(scaler_hr, 'scalers/scaler_hr.pkl')
joblib.dump(scaler_gsr, 'scalers/scaler_gsr.pkl')

print("Scalers fitted and saved")
print("\nPhysiological feature statistics (train):")
print(f"  Heart rate — mean: {scaler_phys.mean_[0]:.2f}, std: {scaler_phys.scale_[0]:.2f}")
print(f"  GSR        — mean: {scaler_phys.mean_[1]:.6f}, std: {scaler_phys.scale_[1]:.6f}")
print("\nTarget statistics (train):")
print(f"  Arousal    — mean: {scaler_arousal.mean_[0]:.4f}, std: {scaler_arousal.scale_[0]:.4f}")
print(f"  Heart rate — mean: {scaler_hr.mean_[0]:.2f}, std: {scaler_hr.scale_[0]:.2f}")
print(f"  GSR        — mean: {scaler_gsr.mean_[0]:.6f}, std: {scaler_gsr.scale_[0]:.6f}")



Scalers fitted and saved

Physiological feature statistics (train):
  Heart rate — mean: 74.20, std: 11.98
  GSR        — mean: 0.000006, std: 0.000007

Target statistics (train):
  Arousal    — mean: 0.4858, std: 0.3111
  Heart rate — mean: 74.20, std: 11.98
  GSR        — mean: 0.000006, std: 0.000007


## Piece 1 — Full-Sequence Dataset

Groups rows by file (one sample per trial) instead of by loop. Each sample is
the full sequence of up to 148 time-bins, padded to 148 with a mask marking
real vs padded steps. Loop number is encoded as a per-timestep one-hot vector
(4-dim), since loop identity now changes within a single sequence (bins 0-37
are loop 1, 38-75 loop 2, and so on). Applies the train-fitted scalers to the
physiological features and the arousal target. Returns the topomap sequence,
phys, per-step loop one-hot, treatment index, target, and mask.

In [5]:
class EMAPDatasetFullSeq(Dataset):
    def __init__(self, labels_df, npy_dir, scaler_phys, scaler_target,
                 target_col='arousal', split='train', max_len=148):
        self.npy_dir = Path(npy_dir)
        self.scaler_phys = scaler_phys
        self.scaler_target = scaler_target
        self.target_col = target_col
        self.max_len = max_len
        self.samples = []

        # ONE sample per file (all 4 loops together)
        for filename, group in labels_df.groupby('filename'):
            group = group.sort_values('row_idx').reset_index(drop=True)
            self.samples.append({
                'filename': filename,
                'treatment_idx': treatment_to_idx[group['treatment_id'].iloc[0]],
                'row_indices': group['row_idx'].values,
                'loop_nums': group['loop_num'].values,
                'phys_features': group[['heartrate', 'gsr']].values,
                'targets': group[target_col].values
            })
        print(f"{split} full-seq dataset: {len(self.samples)} samples (one per file)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        seq = np.load(self.npy_dir / f"{s['filename']}.npy")[s['row_indices']]

        phys_scaled = self.scaler_phys.transform(
            pd.DataFrame(s['phys_features'], columns=['heartrate', 'gsr']))
        targets_scaled = self.scaler_target.transform(
            pd.DataFrame(s['targets'].reshape(-1, 1), columns=[self.target_col])).flatten()

        real_len = seq.shape[0]
        L = self.max_len

        loop_oh = np.zeros((real_len, 4), dtype=np.float32)
        for i, ln in enumerate(s['loop_nums']):
            loop_oh[i, int(ln) - 1] = 1.0

        if real_len < L:
            pad = L - real_len
            seq = np.pad(seq, ((0, pad), (0, 0), (0, 0), (0, 0)))
            phys_scaled = np.pad(phys_scaled, ((0, pad), (0, 0)))
            targets_scaled = np.pad(targets_scaled, (0, pad))
            loop_oh = np.pad(loop_oh, ((0, pad), (0, 0)))
        elif real_len > L:
            seq, phys_scaled = seq[:L], phys_scaled[:L]
            targets_scaled = targets_scaled[:L]
            loop_oh = loop_oh[:L]
            real_len = L

        mask = np.zeros(L, dtype=np.float32)
        mask[:real_len] = 1.0

        return (torch.FloatTensor(seq),
                torch.FloatTensor(phys_scaled),
                torch.FloatTensor(loop_oh),
                torch.LongTensor([s['treatment_idx']]),
                torch.FloatTensor(targets_scaled),
                torch.FloatTensor(mask))

### Instantiate full-sequence datasets

Creates train and val datasets with max_len=148. Sample counts should now
equal the number of files (~2038 train, ~695 val), roughly a quarter of the
per-loop counts, confirming the per-file grouping worked.

In [6]:
train_dataset = EMAPDatasetFullSeq(train_labels, TOPOMAP_DIR/'train'/'npy',
                                   scaler_phys, scaler_arousal,
                                   target_col='arousal', split='train', max_len=148)
val_dataset = EMAPDatasetFullSeq(val_labels, TOPOMAP_DIR/'val'/'npy',
                                 scaler_phys, scaler_arousal,
                                 target_col='arousal', split='val', max_len=148)

train full-seq dataset: 2038 samples (one per file)
val full-seq dataset: 695 samples (one per file)


## Piece 2 — Full-Sequence Model

CNN+GRU adapted for whole-trial input. The CNN encodes each of the up-to-148
timesteps' topomaps; the GRU then processes the full sequence. The key change
from the per-loop model: loop arrives as a per-timestep one-hot of shape
(batch, seq_len, 4) and is concatenated at each timestep directly, rather than
expanded from a single per-sample value. Video-ID embedding stays on
(use_treatment=True) and is expanded across all timesteps. Architecture carries
the tuned step-3 config: cnn_feature_dim=256, hidden_dim=256, 1 GRU layer,
dropout=0.59.

In [7]:
import torch.nn as nn

class CNNEncoder(nn.Module):
    def __init__(self, in_channels=4, feature_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(256, feature_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.encoder(x).flatten(1)
        return self.fc(self.dropout(x))


class EEGArousalModelFullSeq(nn.Module):
    def __init__(self, cnn_feature_dim=256, hidden_dim=256,
                 num_gru_layers=1, dropout=0.59, use_treatment=True):
        super().__init__()
        self.cnn = CNNEncoder(in_channels=4, feature_dim=cnn_feature_dim)
        self.use_treatment = use_treatment
        if use_treatment:
            self.treatment_emb = nn.Embedding(NUM_TREATMENTS, 8)
        extra = 8 if use_treatment else 0
        # CNN features + phys(2) + loop one-hot(4) + treatment(8)
        gru_input_dim = cnn_feature_dim + 2 + 4 + extra

        self.gru = nn.GRU(gru_input_dim, hidden_dim, num_gru_layers,
                          batch_first=True,
                          dropout=dropout if num_gru_layers > 1 else 0.0,
                          bidirectional=True)
        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, topomap, phys, loop_onehot, treatment=None):
        b, seq_len, C, H, W = topomap.shape
        cnn_features = self.cnn(topomap.view(b * seq_len, C, H, W)).view(b, seq_len, -1)

        feats = [cnn_features, phys, loop_onehot]   # loop is already (b, seq_len, 4)
        if self.use_treatment:
            treat = self.treatment_emb(treatment.squeeze(-1)).unsqueeze(1).expand(-1, seq_len, -1)
            feats.append(treat)
        combined = torch.cat(feats, dim=-1)
        gru_out, _ = self.gru(combined)
        return self.prediction_head(gru_out).squeeze(-1)

## Piece 3 — DataLoaders and Shape Verification

Wraps the datasets in DataLoaders at batch_size=16 (reduced from 32, since
148-step sequences use ~4x the memory of single loops). Pulls one real batch
and prints shapes to confirm the full-sequence pipeline is correct before
training: topomap (16, 148, 4, 128, 128), phys (16, 148, 2), per-step loop
one-hot (16, 148, 4), treatment (16, 1), target and mask (16, 148).

In [8]:
BATCH_SIZE = 16
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# ---- Verify one real batch ----
topomap, phys, loop_oh, treatment, target, mask = next(iter(train_loader))
print("\nBatch shapes:")
print(f"  Topomap:   {topomap.shape}")    # (16, 148, 4, 128, 128)
print(f"  Phys:      {phys.shape}")        # (16, 148, 2)
print(f"  Loop OH:   {loop_oh.shape}")     # (16, 148, 4)  <- per-timestep one-hot
print(f"  Treatment: {treatment.shape}")   # (16, 1)
print(f"  Target:    {target.shape}")      # (16, 148)
print(f"  Mask:      {mask.shape}")         # (16, 148)

# quick sanity: a one-hot row should sum to 1 on real steps, 0 on padding
print(f"\n  Loop OH row sums (first sample, first 5 steps): {loop_oh[0,:5].sum(dim=-1).tolist()}")
print(f"  Mask real steps in first sample: {int(mask[0].sum())} of 148")

Train batches: 128  |  Val batches: 44

Batch shapes:
  Topomap:   torch.Size([16, 148, 4, 128, 128])
  Phys:      torch.Size([16, 148, 2])
  Loop OH:   torch.Size([16, 148, 4])
  Treatment: torch.Size([16, 1])
  Target:    torch.Size([16, 148])
  Mask:      torch.Size([16, 148])

  Loop OH row sums (first sample, first 5 steps): [1.0, 1.0, 1.0, 1.0, 1.0]
  Mask real steps in first sample: 100 of 148


## Piece 4 — Instantiate and Train

Instantiates the full-sequence model with the tuned step-3 configuration
(hidden_dim=256, 1 GRU layer, dropout=0.59, video-ID embedding on) and trains
for 50 epochs. Uses the masked first-difference + MSE loss with lambda_diff=0.88
(step-3 value); evaluation stays plain masked RMSE for comparability to prior
runs. Saves the best checkpoint by validation RMSE. This is the first real
memory test at batch_size=16 — if it OOMs, reduce to 8.

In [9]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import math, json, time, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("models", exist_ok=True)

# ---- tuned step-3 config ----
LAMBDA = 0.88
LR = 0.0015
EPOCHS = 50
EXPERIMENT = "step4_fullseq"
DESCRIPTION = "Full-sequence (whole file, max_len=148), tuned step-3 config (h=256, 1 layer, drop=0.59), first-diff loss lambda=0.88, videoID on"
BEST_MODEL_PATH = f"models/{EXPERIMENT}_best.pt"
RESULTS_LOG = "models/results_log.jsonl"

model = EEGArousalModelFullSeq(
    cnn_feature_dim=256, hidden_dim=256,
    num_gru_layers=1, dropout=0.59, use_treatment=True
).to(device)

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

def masked_combined_loss(output, target, mask, lambda_diff=0.88):
    sq_err = ((output - target) ** 2) * mask
    mse_term = sq_err.sum() / mask.sum()
    pred_diff = output[:, 1:] - output[:, :-1]
    targ_diff = target[:, 1:] - target[:, :-1]
    diff_mask = mask[:, 1:] * mask[:, :-1]
    diff_term = (((pred_diff - targ_diff) ** 2) * diff_mask).sum() / diff_mask.sum()
    return mse_term + lambda_diff * diff_term

def evaluate(model, loader):
    model.eval()
    total_loss, preds, targs = 0, [], []
    with torch.no_grad():
        for topomap, phys, loop_oh, treatment, target, mask in loader:
            topomap, phys, loop_oh = topomap.to(device), phys.to(device), loop_oh.to(device)
            treatment, target, mask = treatment.to(device), target.to(device), mask.to(device)
            out = model(topomap, phys, loop_oh, treatment)
            total_loss += (((out - target) ** 2) * mask).sum().item() / mask.sum().item()
            mb = mask.bool()
            preds.append(out[mb].cpu().numpy()); targs.append(target[mb].cpu().numpy())
    preds = scaler_arousal.inverse_transform(np.concatenate(preds).reshape(-1,1)).flatten()
    targs = scaler_arousal.inverse_transform(np.concatenate(targs).reshape(-1,1)).flatten()
    return total_loss/len(loader), math.sqrt(np.mean((preds-targs)**2))

best_val_rmse, best_epoch = float("inf"), 0
print(f"{'Epoch':>6} {'TrainLoss':>12} {'ValLoss':>10} {'ValRMSE':>10} {'LR':>10}")
print("-"*55)
for epoch in range(1, EPOCHS+1):
    model.train()
    total = 0
    for topomap, phys, loop_oh, treatment, target, mask in train_loader:
        topomap, phys, loop_oh = topomap.to(device), phys.to(device), loop_oh.to(device)
        treatment, target, mask = treatment.to(device), target.to(device), mask.to(device)
        optimizer.zero_grad()
        out = model(topomap, phys, loop_oh, treatment)
        loss = masked_combined_loss(out, target, mask, lambda_diff=LAMBDA)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    train_loss = total/len(train_loader)
    val_loss, val_rmse = evaluate(model, val_loader)
    scheduler.step(val_loss)
    if val_rmse < best_val_rmse:
        best_val_rmse, best_epoch = val_rmse, epoch
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "val_rmse": val_rmse}, BEST_MODEL_PATH)
        marker = " *"
    else:
        marker = ""
    lr = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} {train_loss:>12.6f} {val_loss:>10.6f} {val_rmse:>10.6f} {lr:>10.2e}{marker}")

print(f"\nBest Val RMSE: {best_val_rmse:.6f} at epoch {best_epoch}")
with open(RESULTS_LOG, "a") as f:
    f.write(json.dumps({"experiment": EXPERIMENT, "description": DESCRIPTION,
                        "best_val_rmse": round(best_val_rmse,6), "best_epoch": best_epoch,
                        "timestamp": time.strftime("%Y-%m-%d %H:%M")}) + "\n")

 Epoch    TrainLoss    ValLoss    ValRMSE         LR
-------------------------------------------------------
     1     1.012438   1.103073   0.325750   1.50e-03 *
     2     0.988785   1.485961   0.380631   1.50e-03
     3     0.975343   1.057113   0.318220   1.50e-03 *
     4     0.974127   1.013050   0.310887   1.50e-03 *
     5     0.969425   1.094771   0.323549   1.50e-03
     6     0.976850   1.023087   0.312310   1.50e-03
     7     0.969381   1.027697   0.313334   1.50e-03
     8     0.976956   1.016584   0.311430   1.50e-03
     9     0.970099   1.029075   0.313301   1.50e-03
    10     0.967223   1.016903   0.311477   7.50e-04
    11     0.961769   1.022398   0.312289   7.50e-04
    12     0.967174   1.050963   0.316912   7.50e-04
    13     0.966828   1.046793   0.316100   7.50e-04
    14     0.962740   1.046447   0.315942   7.50e-04
    15     0.958557   1.036501   0.314386   7.50e-04
    16     0.955197   1.062150   0.318437   3.75e-04
    17     0.948892   1.093632   0.32